In [1]:
import pandas as pd
from urllib.parse import quote

# The physical and signal features we want to feed our model
FEATURES = [
    'koi_period', 'koi_duration', 'koi_depth', 'koi_prad', 
    'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_srad'
]

# Building the SQL-like query to get the target label (koi_disposition), 
# the star ID (kepid), and our features from the 'cumulative' table
query = f"select kepid, koi_disposition, {', '.join(FEATURES)} from cumulative"

# Encoding the query into the API URL
url = ('https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=' 
       + quote(query) + '&format=csv')

print("Fetching data from NASA Exoplanet Archive...")
df = pd.read_csv(url)

print("Data loaded successfully!")
print(f"Dataset shape: {df.shape}")
df.head()

Fetching data from NASA Exoplanet Archive...
Data loaded successfully!
Dataset shape: (9564, 11)


,kepid,koi_disposition,koi_period,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_srad
0,10797460,CONFIRMED,9.488036,2.95750,615.8,2.26,793.0,93.59,35.8,5455.0,0.927
1,10797460,CONFIRMED,54.418383,4.50700,874.8,2.83,443.0,9.11,25.8,5455.0,0.927
2,10811496,CANDIDATE,19.899140,1.78220,10829.0,14.60,638.0,39.30,76.3,5853.0,0.868
3,10848459,FALSE POSITIVE,1.736952,2.40641,8079.2,33.46,1395.0,891.96,505.6,5805.0,0.791
4,10854555,CONFIRMED,2.525592,1.65450,603.3,2.75,1406.0,926.16,40.9,6031.0,1.046


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Filter out candidates and drop rows with missing data
labeled = df[df.koi_disposition.isin(['CONFIRMED', 'FALSE POSITIVE'])].dropna()

# Define our X (features) and y (target label: 1 for Confirmed, 0 for False Positive)
X = labeled[FEATURES]
y = (labeled.koi_disposition == 'CONFIRMED').astype(int)

# Split by star (kepid), not by row, to prevent data leakage
split = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups=labeled.kepid))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

Training set size: 5871 samples
Testing set size: 1457 samples


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Random Forest model
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Output the performance metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))